# Roots of Equations in Numerical Methods  
### (With Optimization Applications + Exercises + Solutions)

This notebook is a **course note** for the topic **root finding** in numerical methods.  

---

## Learning objectives
By the end, you should be able to:

1. Explain what it means to solve an equation \(f(x)=0\) and why it matters.
2. Implement and use: **Bisection**, **Fixed-Point Iteration**, **Newton–Raphson**, **Secant**.
3. Compare methods (robustness, speed, derivative requirement).
4. Connect root finding to **optimization** by solving \(C'(x)=0\).
5. Solve **real-world exercises** with complete solutions.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

def plot_function(f, a, b, title=None, n=400):
    xs = np.linspace(a, b, n)
    ys = np.array([f(x) for x in xs])
    plt.figure(figsize=(7,4))
    plt.plot(xs, ys)
    plt.axhline(0, linewidth=1)
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(title if title else "Function plot")
    plt.grid(True, alpha=0.2)
    plt.show()


## 1. What is a root and why do we care?

A **root** (or **zero**) of a function \(f\) is a value \(x*\) such that:

\[
f(x*) = 0.
\]

### Where does it appear in real life?

| Domain | Example | Root equation |
|---|---|---|
| Optimization | minimize cost \(C(x)\) | \(C'(x)=0\) |
| Economics | equilibrium price | Demand–Supply \(=0\) |
| Physics | equilibrium position | Net force \(=0\) |
| Engineering | steady state | residual \(=0\) |
| ML / DL | training objective | gradient \(=0\) (or close) |

### Why numerical methods?
Many equations cannot be solved exactly in closed form, so we compute an **approximation** of the root.


## 2. A running example

We will use a classical example:

\[
f(x) = x^3 - 2x - 5.
\]

It has a real root between 2 and 3 (because \(f(2) < 0\) and \(f(3) > 0\)).


In [ ]:
def f1(x):
    return x**3 - 2*x - 5

plot_function(f1, 1, 4, title="Running example: f(x) = x^3 - 2x - 5")
print("f(2) =", f1(2))
print("f(3) =", f1(3))


## 3. Stopping criteria (when do we stop?)

Common stopping rules:

**Stopping criteria**

1. **Function is close to zero**

   $|f(x_k)| \le \varepsilon_f$

2. **Step size is small**

   $|x_{k+1}-x_k| \le \varepsilon_x$

3. **Maximum number of iterations**

   $k \le k_{\max}$



## 4. Method 1 — Bisection (Bracketing method)

### Idea
If $f(a)$ and $f(b)$ have opposite signs, that is,

$$
f(a)\,f(b) < 0
$$

then there exists at least one root in the interval $[a,b]$
(Intermediate Value Theorem).

Bisection repeatedly halves the interval $[a,b]$ while preserving a sign change.  
It is **slow but guaranteed** (if the conditions hold).

### Convergence

After $n$ iterations, the interval length is

$$
\frac{b-a}{2^n}
$$

so the root error is bounded by

$$
|x_n - x^\*| \le \frac{b-a}{2^n}.
$$




In [ ]:
def bisection(f, a, b, tol_x=1e-8, tol_f=1e-10, max_iter=200, return_history=False):
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("Bisection requires f(a) and f(b) to have opposite signs (a bracket).")

    history = []
    for k in range(max_iter):
        m = 0.5*(a + b)
        fm = f(m)

        if return_history:
            history.append((k, a, b, m, fm))

        if abs(fm) <= tol_f or abs(b - a) <= tol_x:
            return (m, history) if return_history else m

        if fa * fm <= 0:
            b, fb = m, fm
        else:
            a, fa = m, fm

    return (m, history) if return_history else m

root_bis, hist_bis = bisection(f1, 2, 3, return_history=True)
print("Bisection root:", root_bis, " |f(root)|=", abs(f1(root_bis)), " iterations=", len(hist_bis))


In [ ]:
iters = [h[0] for h in hist_bis]
fvals = [abs(h[4]) for h in hist_bis]

plt.figure(figsize=(7,4))
plt.semilogy(iters, fvals)
plt.xlabel("Iteration")
plt.ylabel("|f(mid)| (log scale)")
plt.title("Bisection: decrease of |f(x_k)|")
plt.grid(True, alpha=0.2)
plt.show()


## 5. Method 2 — Fixed-Point Iteration

### Idea
Rewrite $f(x)=0$ into

$$
x = g(x)
$$

Then iterate

$$
x_{k+1} = g(x_k)
$$

Convergence is typically guaranteed locally if

$$
\lvert g'(x^\*) \rvert < 1.
$$



In [ ]:
def fixed_point(g, x0, tol_x=1e-8, max_iter=200, return_history=False):
    history = []
    x = float(x0)
    for k in range(max_iter):
        x_new = g(x)
        if return_history:
            history.append((k, x, x_new, abs(x_new - x)))
        if abs(x_new - x) <= tol_x:
            return (x_new, history) if return_history else x_new
        x = x_new
    return (x, history) if return_history else x

def g1(x):
    return (2*x + 5)**(1/3)

x_fp, hist_fp = fixed_point(g1, x0=2.5, return_history=True)
print("Fixed-point root:", x_fp, " |f(root)|=", abs(f1(x_fp)), " iterations=", len(hist_fp))


In [ ]:
errs = [h[3] for h in hist_fp]
plt.figure(figsize=(7,4))
plt.semilogy(range(len(errs)), errs)
plt.xlabel("Iteration")
plt.ylabel("|x_{k+1}-x_k| (log scale)")
plt.title("Fixed-point: step size decay")
plt.grid(True, alpha=0.2)
plt.show()


## 6. Method 3 — Newton–Raphson (Tangent method)

Newton update:

$$
x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}
$$

It is fast near the root, but it needs a good initialization and the derivative.


In [ ]:
def newton(f, df, x0, tol_x=1e-10, tol_f=1e-12, max_iter=50, return_history=False):
    history = []
    x = float(x0)
    for k in range(max_iter):
        fx = f(x)
        dfx = df(x)
        if dfx == 0:
            raise ZeroDivisionError("Newton failed: derivative is zero.")
        x_new = x - fx/dfx

        if return_history:
            history.append((k, x, fx, dfx, x_new, abs(x_new-x)))

        if abs(f(x_new)) <= tol_f or abs(x_new - x) <= tol_x:
            return (x_new, history) if return_history else x_new

        x = x_new
    return (x, history) if return_history else x

def df1(x):
    return 3*x**2 - 2

x_newt, hist_newt = newton(f1, df1, x0=2.5, return_history=True)
print("Newton root:", x_newt, " |f(root)|=", abs(f1(x_newt)), " iterations=", len(hist_newt))


In [ ]:
fvals = [abs(h[2]) for h in hist_newt]
plt.figure(figsize=(7,4))
plt.semilogy(range(len(fvals)), fvals)
plt.xlabel("Iteration")
plt.ylabel("|f(x_k)| (log scale)")
plt.title("Newton: fast decrease of |f(x_k)|")
plt.grid(True, alpha=0.2)
plt.show()


## 7. Method 4 — Secant (Derivative-free Newton)

Secant update:

$$
x_{k+1} = x_k - f(x_k)\frac{x_k-x_{k-1}}{f(x_k)-f(x_{k-1})}.
$$

No derivative needed, often fast, but not always guaranteed.


In [ ]:
def secant(f, x0, x1, tol_x=1e-10, tol_f=1e-12, max_iter=80, return_history=False):
    history = []
    x_prev, x = float(x0), float(x1)

    for k in range(max_iter):
        f_prev, fx = f(x_prev), f(x)
        denom = (fx - f_prev)
        if denom == 0:
            raise ZeroDivisionError("Secant failed: f(x_k) == f(x_{k-1}).")
        x_new = x - fx*(x - x_prev)/denom

        if return_history:
            history.append((k, x_prev, x, fx, x_new, abs(x_new-x)))

        if abs(f(x_new)) <= tol_f or abs(x_new - x) <= tol_x:
            return (x_new, history) if return_history else x_new

        x_prev, x = x, x_new

    return (x, history) if return_history else x

x_sec, hist_sec = secant(f1, 2.0, 3.0, return_history=True)
print("Secant root:", x_sec, " |f(root)|=", abs(f1(x_sec)), " iterations=", len(hist_sec))


## 8. Method comparison on the running example

In [ ]:
results = [
    ("Bisection", root_bis, abs(f1(root_bis)), len(hist_bis)),
    ("Fixed-point", x_fp, abs(f1(x_fp)), len(hist_fp)),
    ("Newton", x_newt, abs(f1(x_newt)), len(hist_newt)),
    ("Secant", x_sec, abs(f1(x_sec)), len(hist_sec)),
]
for name, xhat, res, iters in results:
    print(f"{name:12s}  x ≈ {xhat:.10f}   |f(x)|={res:.3e}   iterations={iters}")


## 9. Real-world application: Market equilibrium (Economics)

Demand: $$D(p)=100-10p $$ 
Supply: $$ S(p)=20+5p $$

Equilibrium solves:
$$
f(p)=D(p)-S(p)=0.
$$


In [ ]:
def f_market(p):
    return 100 - 10*p - (20 + 5*p)

def df_market(p):
    return -15.0

plot_function(f_market, 0, 10, title="Market equilibrium: f(p)=Demand-Supply")

p_bis = bisection(f_market, 0, 10)
p_newt = newton(f_market, df_market, x0=2)
print("Equilibrium price (bisection):", p_bis)
print("Equilibrium price (newton):   ", p_newt)


## 10. Optimization connection 

To minimize $$C(x)=(x-3)^2 + 1 $$, solve:
$$
C'(x)=0.
$$

So 1D optimization becomes **root finding of the derivative**.


In [ ]:
def C(x):
    return (x-3)**2 + 1

def dC(x):
    return 2*(x-3)

def ddC(x):
    return 2.0

x_star = newton(dC, ddC, x0=0.0)
print("Minimum at x* =", x_star, "  C(x*) =", C(x_star))
plot_function(dC, -1, 7, title="Solve C'(x)=0 to find optimum")


## 11. Exercises (with solutions)

### Exercise 1
Solve $$x^3-2x-5=0$$ using bisection, Newton, and secant.


### Exercise 2 (Profit maximization)
Profit: $$P(x)=100x-10x^2$$. Find $$x^*$$ that maximizes profit by solving $$P'(x)=0$$.




### Exercise 3 (Multiple stationary points)
$$
C(x)=x^4-3x^2+2
$$
Solve $$C'(x)=0$$ and classify stationary points with $$C''(x)$$.




## 12. Key takeaways (teach this)

- Root finding solves $$f(x)=0$$ and is **everywhere**.
- **Bisection**: slow but guaranteed (needs sign-change bracket).
- **Newton**: very fast, needs derivative, sensitive to initialization.
- **Secant**: derivative-free, often fast, not guaranteed.
- Optimization in 1D: solve $$C'(x)=0$$.
- Calibration tasks in ML/engineering are also root finding.

